# **🔧 Kurulum ve Ortam Hazırlığı**

Bu bölümde çalışma ortamı hazırlanır:

- Google Drive bağlantısı yapılır.  
- Çalışma dizini (`/MyDrive/NLP/Translation`) aktif hale getirilir.  
- Birleştirilen veri dosyalarının kaydedileceği `merged_datasets` klasörü oluşturulur.  

> Bu proje, dört dil (Arabic, Chinese, French, Italian) için çok dilli duygu analizi veri hazırlama pipeline'ı içermektedir.  
> Her dilin verisi daha sonra çeviri, ön işleme ve modelleme aşamalarına tabi tutulacaktır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.chdir('/content/drive/MyDrive/NLP/Translation')
!ls

'arabic dataset'     'french dataset'	 sentiment_gpt3_train.jsonl
'chinese dataset'     get-pip.py	'Translation v1.ipynb'
 Datasets.ipynb       get-pip.py.1	'Translation v2.ipynb'
'Datasets v2.ipynb'  'italian dataset'
 final_models	      merged_datasets


In [ ]:
base_dir = '/content/drive/MyDrive/NLP/Translation'

merged_dir = os.path.join(base_dir, 'merged_datasets')
os.makedirs(merged_dir, exist_ok=True)

print("✅ Google Drive bağlandı ve çalışma dizini ayarlandı.")
print(f"📂 Aktif dizin: {os.getcwd()}")

print("\n📁 İçerik:")
!ls "$base_dir"

✅ Google Drive bağlandı ve çalışma dizini ayarlandı.
📂 Aktif dizin: /content/drive/MyDrive/NLP/Translation

📁 İçerik:
'arabic dataset'   'Datasets v2.ipynb'	 merged_datasets
'chinese dataset'  'french dataset'	'Translation v1.ipynb'
 Datasets.ipynb    'italian dataset'


#  2️⃣ Arabic Dataset Dönüştürme ve CSV’ye Kaydetme

Bu bölümde, Arapça duygu analizi verisini (`Tweets.txt`) okunabilir bir **CSV** formatına dönüştürüyoruz.  
Veri iki sütundan oluşmaktadır:

- `text`: Tweet metni  
- `label`: Duygu etiketi (`POS`, `NEG`, `NEUTRAL`, `OBJ`)

Dönüştürme adımlarında:

1. Veriyi `\t` (tab) ayracıyla okuyoruz.  
2. Gereksiz boşlukları temizliyoruz.  
3. Yeni bir sütun olarak `language = 'ar'` ekliyoruz.  
4. Veriyi `/merged_datasets/arabic_merged.csv` dosyasına kaydediyoruz.

Bu format, ileride tüm dillerin tek bir veri yapısında birleştirilmesini kolaylaştıracaktır.


In [ ]:
import pandas as pd

input_path = '/content/drive/MyDrive/NLP/Translation/arabic dataset/Tweets.txt'
output_path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged.csv'

df_ar = pd.read_csv(
    input_path,
    sep='\t',
    header=None,
    names=['text', 'label'],
    encoding='utf-8',
    on_bad_lines='skip'
)

df_ar['language'] = 'ar'

df_ar['text'] = df_ar['text'].astype(str).str.strip()
df_ar['label'] = df_ar['label'].astype(str).str.strip()

df_ar.to_csv(output_path, index=False, encoding='utf-8')

print("✅ Arabic dataset başarıyla dönüştürüldü ve kaydedildi.")
print(f"📊 Toplam örnek sayısı: {len(df_ar)}")
print("\n🔍 İlk 5 satır:")
print(df_ar.head())

✅ Arabic dataset başarıyla dönüştürüldü ve kaydedildi.
📊 Toplam örnek sayısı: 9694

🔍 İlk 5 satır:
                                                text    label language
0  بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...      OBJ       ar
1  أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...      POS       ar
2  البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...      NEG       ar
3  #الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...      OBJ       ar
4  الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...  NEUTRAL       ar


# 📊 Arabic Dataset – Sınıf Dağılımı ve OBJ Etiketlerinin Kaldırılması

Bu adımda:
1. Mevcut `arabic_merged.csv` dosyasındaki etiket dağılımı incelenir.  
2. `OBJ` etiketi (nesnel veya sınıflandırılamayan örnekler) çıkarılır.  
3. Kalan veri (`POS`, `NEG`, `NEUTRAL`) yeni bir dosya olarak kaydedilir:


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged.csv'
df_ar = pd.read_csv(path)

class_counts = (
    df_ar['label']
    .value_counts()
    .rename_axis('label')
    .reset_index(name='adet')
)
class_counts['oran (%)'] = (class_counts['adet'] / len(df_ar) * 100).round(2)

print("📊 Sınıf Dağılımı (Tüm Veri):\n")
print(class_counts.to_string(index=False))

df_ar_clean = df_ar[df_ar['label'] != 'OBJ'].copy()

clean_path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_clean.csv'
df_ar_clean.to_csv(clean_path, index=False, encoding='utf-8')

print(f"\n🧹 OBJ etiketli örnekler çıkarıldı.")
print(f"Yeni örnek sayısı: {len(df_ar_clean)}")
print(f"📁 Kaydedildi: {clean_path}")

print("\n📊 Yeni Sınıf Dağılımı (OBJ hariç):")
print(df_ar_clean['label'].value_counts())

📊 Sınıf Dağılımı (Tüm Veri):

  label  adet  oran (%)
    OBJ  6470     66.74
    NEG  1642     16.94
NEUTRAL   805      8.30
    POS   777      8.02

🧹 OBJ etiketli örnekler çıkarıldı.
Yeni örnek sayısı: 3224
📁 Kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_clean.csv

📊 Yeni Sınıf Dağılımı (OBJ hariç):
label
NEG        1642
NEUTRAL     805
POS         777
Name: count, dtype: int64


# 3️⃣ Chinese Dataset – Birleştirme ve CSV’ye Dönüştürme

Bu bölümde, Çince veri kümesinin üç ayrı dosyası birleştirilir:

- `train.jsonl`
- `validation.jsonl`
- `test.jsonl`

Her satır, JSON formatında bir örnektir ve aşağıdaki alanları içerir:
- `id`: örnek kimliği  
- `text`: metin içeriği  
- `label`: duygu etiketi (`0–4` arası değerler)  

İşlem adımları:
1. Üç dosya da sırayla okunur.  
2. Her satırdan `id`, `text`, `label` bilgisi çıkarılır.  
3. `language = 'zh'` sütunu eklenir.  
4. Tüm örnekler tek DataFrame’de birleştirilir ve `chinese_merged.csv` dosyası olarak kaydedilir.  

Bu birleşik veri, model eğitiminde kullanılacak nihai Çince duygu analizi veri kümesini oluşturur.


In [ ]:
import pandas as pd
import json
import glob

base_path = '/content/drive/MyDrive/NLP/Translation/chinese dataset'
output_path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged.csv'

files = sorted(glob.glob(f"{base_path}/*.jsonl"))
print("📂 Bulunan dosyalar:", files)

data = []

for file in files:
    with open(file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            try:
                item = json.loads(line)
                text = item.get('text', '').strip()
                label = str(item.get('label', item.get('label_text', ''))).strip()
                _id = item.get('id', '')
                if text and label:
                    data.append({
                        'id': _id,
                        'text': text,
                        'label': label,
                        'language': 'zh'
                    })
            except json.JSONDecodeError:
                continue

df_zh = pd.DataFrame(data)

df_zh.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Chinese dataset başarıyla birleştirildi ve kaydedildi.")
print(f"Toplam örnek sayısı: {len(df_zh)}")
print(f"📁 Kaydedilen dosya: {output_path}")

print("\n🔍 İlk 5 satır:")
print(df_zh.head())

📂 Bulunan dosyalar: ['/content/drive/MyDrive/NLP/Translation/chinese dataset/test.jsonl', '/content/drive/MyDrive/NLP/Translation/chinese dataset/train.jsonl', '/content/drive/MyDrive/NLP/Translation/chinese dataset/validation.jsonl']
✅ Chinese dataset başarıyla birleştirildi ve kaydedildi.
Toplam örnek sayısı: 210000
📁 Kaydedilen dosya: /content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged.csv

🔍 İlk 5 satır:
           id                                               text label  \
0  zh_0932578  假货\n\n买的两百多的，不是真货，和真的对比了小一圈！特别不好跟30多元的没区别，退货了！...     0   
1  zh_0119456  打开吓一跳！\n\n怎么包装盒都没有，就一个塑料袋装了几张面膜，一看就掉了好几个档次了，都不好用了     0   
2  zh_0070808          垃圾车\n\n差到几点的电池！充电一天！用不到一个小时就没电！怎么给小孩带来快乐？     0   
3  zh_0583904         评价质量很差 。 没几天就坏了 。\n\n没用几天就坏了。 寄回厂家修理也i没修好。     0   
4  zh_0765995  几百块买了个垃圾\n\n垃圾中的垃圾，气死我了，买下不到一个月就充不进去电，联系了客服2次，...     0   

  language  
0       zh  
1       zh  
2       zh  
3       zh  
4       zh  


# Chinese Dataset – Sınıf Dağılımı ve 3 Sınıfa Dönüştürme

Bu adımda:
1. `chinese_merged.csv` dosyasındaki 5 sınıflı duygu etiketleri (`0–4`) incelenir.  
2. Etiketler üç kategoriye indirgenir:
   - `0` veya `1` → **NEGATIVE**
   - `2` → **NEUTRAL**
   - `3` veya `4` → **POSITIVE**
3. Yeni etiket dağılımı hesaplanır ve verisetine `label_3class` sütunu eklenir.  
4. Dönüştürülmüş veri `chinese_merged_3class.csv` olarak kaydedilir.


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged.csv'
df_zh = pd.read_csv(path)

class_counts = (
    df_zh['label']
    .value_counts()
    .rename_axis('label')
    .reset_index(name='adet')
)
class_counts['oran (%)'] = (class_counts['adet'] / len(df_zh) * 100).round(2)

print("📊 Orijinal Sınıf Dağılımı (5 sınıf):\n")
print(class_counts.to_string(index=False))

mapping = {
    '0': 'NEG',
    '1': 'NEG',
    '2': 'NEUTRAL',
    '3': 'POS',
    '4': 'POS'
}
df_zh['label_3class'] = df_zh['label'].astype(str).map(mapping)

new_counts = (
    df_zh['label_3class']
    .value_counts()
    .rename_axis('label_3class')
    .reset_index(name='adet')
)
new_counts['oran (%)'] = (new_counts['adet'] / len(df_zh) * 100).round(2)

print("\n📊 Yeni Sınıf Dağılımı (3 sınıf):\n")
print(new_counts.to_string(index=False))

output_path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class.csv'
df_zh.to_csv(output_path, index=False, encoding='utf-8')

print(f"\n✅ Yeni 3 sınıflı dosya kaydedildi: {output_path}")
print(f"Toplam örnek sayısı: {len(df_zh)}")

📊 Orijinal Sınıf Dağılımı (5 sınıf):

 label  adet  oran (%)
     0 42000      20.0
     1 42000      20.0
     2 42000      20.0
     3 42000      20.0
     4 42000      20.0

📊 Yeni Sınıf Dağılımı (3 sınıf):

label_3class  adet  oran (%)
         NEG 84000      40.0
         POS 84000      40.0
     NEUTRAL 42000      20.0

✅ Yeni 3 sınıflı dosya kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class.csv
Toplam örnek sayısı: 210000


# 4️⃣ French Dataset – Birleştirme ve 3 Sınıfa Dönüştürme

Bu bölümde Fransızca veri seti işlenir. Veriler üç dosyadan okunur:
- `train.jsonl`
- `validation.jsonl`
- `test.jsonl`

Her satır JSON biçimindedir ve şu alanları içerir:
- `id`: örnek kimliği  
- `text`: inceleme metni  
- `label`: duygu etiketi (`0–4` arası değerler)

İşlem adımları:
1. Üç dosya birleştirilir.  
2. Etiketler 3 sınıfa dönüştürülür:
   - `0`–`1` → **NEGATIVE**
   - `2` → **NEUTRAL**
   - `3`–`4` → **POSITIVE**
3. `language = 'fr'` sütunu eklenir.  
4. Nihai veri kümesi `french_merged_3class.csv` olarak kaydedilir.

In [ ]:
import pandas as pd
import json
import glob

base_path = '/content/drive/MyDrive/NLP/Translation/french dataset'
output_path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/french_merged_3class.csv'

files = sorted(glob.glob(f"{base_path}/*.jsonl"))
print("📂 Bulunan dosyalar:", files)

data = []

for file in files:
    with open(file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            try:
                item = json.loads(line)
                text = item.get('text', '').strip()
                label = str(item.get('label', item.get('label_text', ''))).strip()
                _id = item.get('id', '')
                if text and label:
                    data.append({
                        'id': _id,
                        'text': text,
                        'label': label,
                        'language': 'fr'
                    })
            except json.JSONDecodeError:
                continue

df_fr = pd.DataFrame(data)

class_counts = (
    df_fr['label']
    .value_counts()
    .rename_axis('label')
    .reset_index(name='adet')
)
class_counts['oran (%)'] = (class_counts['adet'] / len(df_fr) * 100).round(2)

print("📊 Orijinal Sınıf Dağılımı (5 sınıf):\n")
print(class_counts.to_string(index=False))

mapping = {
    '0': 'NEG',
    '1': 'NEG',
    '2': 'NEUTRAL',
    '3': 'POS',
    '4': 'POS'
}
df_fr['label_3class'] = df_fr['label'].astype(str).map(mapping)

new_counts = (
    df_fr['label_3class']
    .value_counts()
    .rename_axis('label_3class')
    .reset_index(name='adet')
)
new_counts['oran (%)'] = (new_counts['adet'] / len(df_fr) * 100).round(2)

print("\n📊 Yeni Sınıf Dağılımı (3 sınıf):\n")
print(new_counts.to_string(index=False))

df_fr.to_csv(output_path, index=False, encoding='utf-8')

print(f"\n✅ French dataset başarıyla birleştirildi ve 3 sınıfa dönüştürüldü.")
print(f"📁 Kaydedilen dosya: {output_path}")
print(f"Toplam örnek sayısı: {len(df_fr)}")

print("\n🔍 İlk 5 satır:")
print(df_fr.head())

📂 Bulunan dosyalar: ['/content/drive/MyDrive/NLP/Translation/french dataset/test.jsonl', '/content/drive/MyDrive/NLP/Translation/french dataset/train.jsonl', '/content/drive/MyDrive/NLP/Translation/french dataset/validation.jsonl']
📊 Orijinal Sınıf Dağılımı (5 sınıf):

label  adet  oran (%)
    0 42000      20.0
    1 42000      20.0
    2 42000      20.0
    3 42000      20.0
    4 42000      20.0

📊 Yeni Sınıf Dağılımı (3 sınıf):

label_3class  adet  oran (%)
         NEG 84000      40.0
         POS 84000      40.0
     NEUTRAL 42000      20.0

✅ French dataset başarıyla birleştirildi ve 3 sınıfa dönüştürüldü.
📁 Kaydedilen dosya: /content/drive/MyDrive/NLP/Translation/merged_datasets/french_merged_3class.csv
Toplam örnek sayısı: 210000

🔍 İlk 5 satır:
           id                                               text label  \
0  fr_0797681  ARNAQUE\n\nCa c'est du vol... Vous ne recevrez...     0   
1  fr_0329266  NON REÇU!!!!\n\nPresque 1 mois après la comman...     0   
2  fr_0745602

# Italian Dataset – Sütun Yapısının İncelenmesi

Dönüştürme adımlarına geçmeden önce, İtalyanca verisetindeki sütun adlarını ve örnek satırları inceliyoruz.  
Bu adım, hangi sütunların duygu (pozitif / negatif) etiketlerini içerdiğini ve veri yapısının nasıl olduğunu anlamamızı sağlar.  

Veri dosyası:
- `sentipolc16_officialdistrib_train.csv`

Aşağıdaki hücre:
- toplam satır sayısını,  
- sütun adlarını,  
- ilk birkaç satırı  
gösterir.

In [ ]:
import pandas as pd

train_path = '/content/drive/MyDrive/NLP/Translation/italian dataset/sentipolc16_officialdistrib_train.csv'

df_train = pd.read_csv(train_path, sep=',', encoding='utf-8', on_bad_lines='skip')

print(f"📄 Train veri boyutu: {len(df_train)} satır, {len(df_train.columns)} sütun")

print("\n📊 Train sütunları:")
print(df_train.columns.tolist())

print("\n🔍 Train verisinden ilk 3 satır:")
print(df_train.head(3))

📄 Train veri boyutu: 7410 satır, 9 sütun

📊 Train sütunları:
['idtwitter', 'subj', 'opos', 'oneg', 'iro', 'lpos', 'lneg', 'top', 'text']

🔍 Train verisinden ilk 3 satır:
            idtwitter  subj  opos  oneg  iro  lpos  lneg  top  \
0  122449983151669248     1     0     1    0     0     1    1   
1  125485104863780865     1     0     1    0     0     1    1   
2  125513454315507712     1     0     1    0     0     1    1   

                                                text  
0  Intanto la partita per Via Nazionale si compli...  
1  False illusioni, sgradevoli realtà Mario Monti...  
2  False illusioni, sgradevoli realtà #editoriale...  


# Italian Dataset – 3 Sınıfa Dönüştürme (POS / NEG / NEUTRAL)

Bu aşamada:
1. `opos` (pozitif duygu) ve `oneg` (negatif duygu) sütunları kullanılarak yeni bir `sentiment` sütunu oluşturulur.  
2. Dört olasılıktan oluşan yapı üç sınıfa indirgenir:
   - `1,0` → **POS**
   - `0,1` → **NEG**
   - `1,1` veya `0,0` → **NEUTRAL**
3. Gereksiz sütunlar (id, subj, iro, lpos, lneg, top vb.) kaldırılır.  
4. Sadece `text` ve `sentiment` sütunları bırakılarak temiz veri kaydedilir.


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/NLP/Translation/italian dataset/sentipolc16_officialdistrib_train.csv'
df = pd.read_csv(path, sep=',', encoding='utf-8', on_bad_lines='skip')

df['sentiment'] = 'NEUTRAL'
df.loc[(df['opos'] == 1) & (df['oneg'] == 0), 'sentiment'] = 'POS'
df.loc[(df['oneg'] == 1) & (df['opos'] == 0), 'sentiment'] = 'NEG'

df_clean = df[['text', 'sentiment']].copy()
df_clean['language'] = 'it'

class_counts = (
    df_clean['sentiment']
    .value_counts()
    .rename_axis('label')
    .reset_index(name='adet')
)
class_counts['oran (%)'] = (class_counts['adet'] / len(df_clean) * 100).round(2)

output_path = '/content/drive/MyDrive/NLP/Translation/merged_datasets/italian_merged_3class.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ İtalyanca verisetinde 3 sınıf (POS, NEG, NEUTRAL) başarıyla oluşturuldu.")
print(f"📁 Kaydedilen dosya: {output_path}")
print(f"📊 Toplam örnek sayısı: {len(df_clean)}\n")
print("📊 Sınıf Dağılımı:\n")
print(class_counts.to_string(index=False))

print("\n🔍 İlk 5 satır:")
print(df_clean.head())

✅ İtalyanca verisetinde 3 sınıf (POS, NEG, NEUTRAL) başarıyla oluşturuldu.
📁 Kaydedilen dosya: /content/drive/MyDrive/NLP/Translation/merged_datasets/italian_merged_3class.csv
📊 Toplam örnek sayısı: 7410

📊 Sınıf Dağılımı:

  label  adet  oran (%)
NEUTRAL  3256     43.94
    NEG  2543     34.32
    POS  1611     21.74

🔍 İlk 5 satır:
                                                text sentiment language
0  Intanto la partita per Via Nazionale si compli...       NEG       it
1  False illusioni, sgradevoli realtà Mario Monti...       NEG       it
2  False illusioni, sgradevoli realtà #editoriale...       NEG       it
3  Mario Monti: Berlusconi risparmi all'Italia il...       NEG       it
4  Mario Monti: Berlusconi risparmi all'Italia il...       NEG       it


# **🧹 Veri Temizleme (Preprocessing) Aşaması**

Bu bölümde tüm diller için ortak bir temizlik fonksiyonu tanımlanır.  
Amaç, veriyi model eğitimine hazır hale getirmektir.  

Aşağıdaki adımlar uygulanır:

✅ **Null ve boş kayıtların kaldırılması**  
✅ **Emojilerin silinmesi**  
✅ **Noktalama işaretlerinin kaldırılması**  
✅ **Küçük harfe çevirme**  
✅ **URL, mention, hashtag ve sayı temizliği**  
✅ **Fazla boşlukların temizlenmesi**  
✅ **Yinelenen satırların kaldırılması**  

Bu işlemler sonucunda her dil için daha temiz, dengeli ve işlenebilir bir veri kümesi elde edilir.

In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import emoji

def clean_text(text):
    """Tek bir metin girdisini temizler."""
    if pd.isna(text):
        return None

    text = text.lower()

    text = re.sub(r'http\S+|www\.\S+|@\w+', '', text)

    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\d+', '', text)

    text = emoji.replace_emoji(text, replace='')

    text = re.sub(r'[^\w\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


def preprocess_dataset(file_path, output_path):
    """
    Verilen CSV dosyasını temizler ve kaydeder.
    Uygulanan işlemler:
    - Null kayıtları atar
    - Yinelenen satırları kaldırır
    - clean_text() fonksiyonunu uygular
    """
    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')

    df = df.dropna(subset=['text'])

    df = df.drop_duplicates(subset=['text'])

    df['text'] = df['text'].apply(clean_text)

    df = df.dropna(subset=['text'])
    df = df[df['text'].str.strip() != '']

    df.to_csv(output_path, index=False, encoding='utf-8')

    print(f"✅ Temizlendi ve kaydedildi: {output_path}")
    print(f"🧾 Yeni veri boyutu: {len(df)} satır")
    print(f"📊 Sütunlar: {df.columns.tolist()}")

# 🧽 Tüm Diller İçin Toplu Veri Temizleme

Bu adımda tanımlanan `preprocess_dataset()` fonksiyonu her bir dildeki verisete uygulanır.  
Her dosya için şu işlemler yapılır:

- Null ve boş satırların kaldırılması  
- Yinelenen kayıtların silinmesi  
- URL, emoji, hashtag, sayı ve noktalama temizliği  
- Küçük harfe çevirme  
- Fazla boşlukların giderilmesi  

Temizlenmiş veri setleri, `merged_datasets` klasörüne  
`*_clean_final.csv` adıyla kaydedilecektir.

In [ ]:
datasets = {
    "Arabic":  "/content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class.csv",
    "Chinese": "/content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class.csv",
    "French":  "/content/drive/MyDrive/NLP/Translation/merged_datasets/french_merged_3class.csv",
    "Italian": "/content/drive/MyDrive/NLP/Translation/merged_datasets/italian_merged_3class.csv"
}

for lang, path in datasets.items():
    output_path = path.replace(".csv", "_clean_final.csv")
    print(f"\n==============================")
    print(f"🧹 Temizleniyor: {lang} dataset")
    print(f"📁 Girdi: {path}")

    preprocess_dataset(path, output_path)

    print(f"✅ Kaydedildi: {output_path}")
    print("==============================")


🧹 Temizleniyor: Arabic dataset
📁 Girdi: /content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class.csv
✅ Temizlendi ve kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class_clean_final.csv
🧾 Yeni veri boyutu: 3176 satır
📊 Sütunlar: ['text', 'sentiment', 'language']
✅ Kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class_clean_final.csv

🧹 Temizleniyor: Chinese dataset
📁 Girdi: /content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class.csv
✅ Temizlendi ve kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class_clean_final.csv
🧾 Yeni veri boyutu: 207205 satır
📊 Sütunlar: ['text', 'sentiment', 'language']
✅ Kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class_clean_final.csv

🧹 Temizleniyor: French dataset
📁 Girdi: /content/drive/MyDrive/NLP/Translation/merged_datasets/french_merged_3class.csv
✅ Temizlendi ve kaydedild

# 📊 Temizlenmiş Verilerin Analizi

Bu adımda her dil için temizlenmiş verisetleri incelenir.  
Aşağıdaki bilgiler gösterilir:

- Toplam örnek sayısı  
- Sütun adları  
- Sınıf dağılımı  
- İlk 3 örnek metin  

Amaç, temizlik sonrası veri bütünlüğünü ve dengesini doğrulamaktır.

In [ ]:
import pandas as pd

clean_datasets = {
    "Arabic":  "/content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class_clean_final.csv",
    "Chinese": "/content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class_clean_final.csv",
    "French":  "/content/drive/MyDrive/NLP/Translation/merged_datasets/french_merged_3class_clean_final.csv",
    "Italian": "/content/drive/MyDrive/NLP/Translation/merged_datasets/italian_merged_3class_clean_final.csv"
}

for lang, path in clean_datasets.items():
    print("=" * 80)
    print(f"🌍 {lang.upper()} DATASET")
    print(f"📁 {path}")

    df = pd.read_csv(path, encoding='utf-8', on_bad_lines='skip')

    print(f"✅ Toplam örnek sayısı: {len(df)}")
    print(f"📊 Sütunlar: {df.columns.tolist()}")

    class_counts = df['sentiment'].value_counts().rename_axis('class').reset_index(name='adet')
    class_counts['oran (%)'] = (class_counts['adet'] / len(df) * 100).round(2)
    print("\n📊 Sınıf Dağılımı:")
    print(class_counts.to_string(index=False))

    print("\n🔍 Örnek Satırlar:")
    print(df[['text', 'sentiment', 'language']].head(3).to_string(index=False))
    print()

🌍 ARABIC DATASET
📁 /content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class_clean_final.csv
✅ Toplam örnek sayısı: 3176
📊 Sütunlar: ['text', 'sentiment', 'language']

📊 Sınıf Dağılımı:
  class  adet  oran (%)
    NEG  1627     51.23
NEUTRAL   789     24.84
    POS   760     23.93

🔍 Örnek Satırlar:
                                                                                                                         text sentiment language
                                                       أهنئ الدكتور أحمد جمال الدين القيادي بحزب مصر بمناسبة صدور أولى روايته       POS       ar
                                                       البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام العريان الي واشنطن شئ مقرف       NEG       ar
الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقولها ملل الله وكيلك تعطيني محاضرة عن الفسق والفجور بجنوب الشيشان كذا يانبع الحنان   NEUTRAL       ar

🌍 CHINESE DATASET
📁 /content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class_cle

# **🌍 Dillerin Birleştirilmesi (Multilingual Dataset Merging)**

Bu aşamada, daha önce ayrı ayrı temizlenmiş olan dört veri seti **tek bir bütünleşik veri kümesinde** birleştirilir.  
Amaç, tüm dillerdeki verileri tek bir formata dönüştürerek sonraki aşama olan **İngilizceye çeviri (Translation Step)** için hazır hale getirmektir.

## **📦 Girdi Dosyaları**
Her biri üç sütun içermektedir: `text`, `sentiment`, `language`.

| Dil | Dosya Adı | Örnek Sayısı |
|------|-------------|----------------|
| 🇸🇦 Arabic | `arabic_merged_3class_clean_final.csv` | 3176 |
| 🇨🇳 Chinese | `chinese_merged_3class_clean_final.csv` | 207205 |
| 🇫🇷 French | `french_merged_3class_clean_final.csv` | 209585 |
| 🇮🇹 Italian | `italian_merged_3class_clean_final.csv` | 7400 |

## **🧩 Hedef**
Tüm bu dosyalar aşağıdaki biçimde tek bir CSV dosyasında birleştirilecektir:

| text | sentiment | language |
|------|------------|-----------|
| ...Arapça cümle... | NEG | ar |
| ...Çince cümle... | POS | zh |
| ...Fransızca cümle... | NEUTRAL | fr |
| ...İtalyanca cümle... | NEG | it |



In [ ]:
import pandas as pd

paths = {
    "ar": "/content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class_clean_final.csv",
    "zh": "/content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class_clean_final.csv",
    "fr": "/content/drive/MyDrive/NLP/Translation/merged_datasets/french_merged_3class_clean_final.csv",
    "it": "/content/drive/MyDrive/NLP/Translation/merged_datasets/italian_merged_3class_clean_final.csv"
}

dfs = []
for lang, path in paths.items():
    df = pd.read_csv(path, encoding="utf-8")
    dfs.append(df)
    print(f"✅ {lang.upper()} yüklendi: {len(df)} satır")

merged_df = pd.concat(dfs, ignore_index=True)

merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

output_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_all_languages_clean.csv"
merged_df.to_csv(output_path, index=False, encoding="utf-8")

print("\n💾 Tüm diller başarıyla birleştirildi!")
print(f"📁 Kaydedilen dosya: {output_path}")
print(f"📊 Toplam satır sayısı: {len(merged_df)}")

print("\n🌍 Dil Bazında Örnek Sayıları:")
print(merged_df['language'].value_counts())

print("\n🔍 İlk 5 satır:")
print(merged_df.head())

✅ AR yüklendi: 3176 satır
✅ ZH yüklendi: 207205 satır
✅ FR yüklendi: 209585 satır
✅ IT yüklendi: 7400 satır

💾 Tüm diller başarıyla birleştirildi!
📁 Kaydedilen dosya: /content/drive/MyDrive/NLP/Translation/merged_datasets/merged_all_languages_clean.csv
📊 Toplam satır sayısı: 427366

🌍 Dil Bazında Örnek Sayıları:
language
fr    209585
zh    207205
it      7400
ar      3176
Name: count, dtype: int64

🔍 İlk 5 satır:
                                                text sentiment language
0  un ème tome incroyable que de rebondissements ...       POS       fr
1                        漏水要退货或者换货 水杯漏水要退货或者换货请尽快联系我       NEG       zh
2       不错的意面 买了两把味道不错 这款意面煮时间长点后相比较要软一点比较适合本人和孩子的口味       POS       zh
3  plutôt pour jouer jai acheté ce micro pour ani...   NEUTRAL       fr
4  nick mason pink floyd complète une collection ...       POS       fr


# **Her Dilden 3000 Rastgele Örnek ile 12000 Verilik Veriseti**

In [ ]:
# ===============================================================
# ⚖️ Dengeleme: Her Dilden 3000 Rastgele Örnek
# ===============================================================

import pandas as pd

# Dosya yolları
paths = {
    "ar": "/content/drive/MyDrive/NLP/Translation/merged_datasets/arabic_merged_3class_clean_final.csv",
    "zh": "/content/drive/MyDrive/NLP/Translation/merged_datasets/chinese_merged_3class_clean_final.csv",
    "fr": "/content/drive/MyDrive/NLP/Translation/merged_datasets/french_merged_3class_clean_final.csv",
    "it": "/content/drive/MyDrive/NLP/Translation/merged_datasets/italian_merged_3class_clean_final.csv"
}

# Hedef örnek sayısı
N = 3000
dfs = []

for lang, path in paths.items():
    df = pd.read_csv(path, encoding="utf-8")

    # Eğer satır sayısı 3000'den azsa hepsini al
    if len(df) < N:
        sampled = df.copy()
        print(f"⚠️ {lang.upper()} yalnızca {len(df)} satır içeriyor, tamamı alındı.")
    else:
        sampled = df.sample(n=N, random_state=42)
        print(f"✅ {lang.upper()} → {len(sampled)} rastgele örnek seçildi ({len(df)} toplamdan).")

    dfs.append(sampled)

# Birleştir ve karıştır
balanced_df = pd.concat(dfs, ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Kaydet
output_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced.csv"
balanced_df.to_csv(output_path, index=False, encoding="utf-8")

# Özet Bilgiler
print("\n💾 Dengeli veri kümesi oluşturuldu!")
print(f"📁 Kaydedilen dosya: {output_path}")
print(f"📊 Toplam örnek sayısı: {len(balanced_df)}")

print("\n🌍 Dil bazında örnek sayıları:")
print(balanced_df['language'].value_counts())

print("\n🔍 İlk 5 satır:")
print(balanced_df.head())

# **mBART Çeviri**

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from tqdm import tqdm
import pandas as pd
import torch
import time

input_path  = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced.csv"
output_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_mbart.csv"


model_name = "facebook/mbart-large-50-many-to-one-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name).to("cuda")


batch_size = 16
max_length = 512
save_every = 5000
target_lang = "en_XX"
tokenizer.tgt_lang = target_lang


df = pd.read_csv(input_path)
print(f"✅ Veri yüklendi: {len(df)} satır")


lang_map = {
    "ar": "ar_AR",
    "zh": "zh_CN",
    "fr": "fr_XX",
    "it": "it_IT"
}


translated_texts = []

for i in tqdm(range(0, len(df), batch_size), desc="Translating with mBART-50", ncols=100):
    batch = df.iloc[i:i+batch_size]

    src_langs = [lang_map.get(lang, "en_XX") for lang in batch["language"]]
    tokenizer.src_lang = src_langs[0]

    inputs = tokenizer(
        batch["text"].tolist(),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to("cuda")

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[target_lang]
        )

    translations = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    translated_texts.extend(translations)

    if (i + batch_size) % save_every == 0:
        temp = df.iloc[:i+batch_size].copy()
        temp["translated_text"] = translated_texts
        temp.to_csv(output_path, index=False, encoding="utf-8")
        print(f"💾 Ara kayıt yapıldı: {i+batch_size}/{len(df)} satır işlendi ({time.strftime('%H:%M:%S')})")

df["translated_text"] = translated_texts
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"\n✅ Çeviri tamamlandı ve kaydedildi: {output_path}")
print(f"📊 Toplam çevrilen satır: {len(df)}")

✅ Veri yüklendi: 12000 satır


Translating with mBART-50:  83%|██████████████████████████▋     | 625/750 [1:25:27<16:58,  8.15s/it]

💾 Ara kayıt yapıldı: 10000/12000 satır işlendi (19:03:20)


Translating with mBART-50: 100%|████████████████████████████████| 750/750 [1:43:14<00:00,  8.26s/it]


✅ Çeviri tamamlandı ve kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_mbart.csv
📊 Toplam çevrilen satır: 12000


## 🌐 Çeviri 2 — Google NMT (deep_translator)
Bu aşamada, `deep_translator` kütüphanesi ile Google Translate API’si üzerinden çeviri yapılır.  
Her satırın dili (`language` sütunu) tespit edilerek İngilizceye çevrilir.

**Uygulama detayları:**
- Kaynak dosya: `merged_balanced.csv` (12 000 örnek)
- Hedef dil: İngilizce
- Model: Google Neural Machine Translation (Google NMT)
- Batch = 1 (API istek limiti nedeniyle)
- Her 1 000 satırda bir ara kayıt yapılır
- Çıktı: `merged_balanced_translated_google.csv`

Bu sonuç daha sonra mBART-50 ve LibreTranslate çevirileriyle karşılaştırılacaktır.


In [ ]:
from deep_translator import GoogleTranslator
from tqdm import tqdm
import pandas as pd
import time

input_path  = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced.csv"
output_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_google.csv"

df = pd.read_csv(input_path)
print(f"✅ Veri yüklendi: {len(df)} satır")

translated_texts = []

for i in tqdm(range(len(df)), desc="Translating with Google NMT"):
    text = str(df.loc[i, "text"])
    lang = df.loc[i, "language"]

    try:
        translated = GoogleTranslator(source=lang, target="en").translate(text)
    except Exception:
        translated = ""
    translated_texts.append(translated)

    if (i + 1) % 1000 == 0:
        temp = df.iloc[:i+1].copy()
        temp["translated_text"] = translated_texts
        temp.to_csv(output_path, index=False, encoding="utf-8")
        print(f"💾 Ara kayıt yapıldı: {i+1}/{len(df)} satır ({time.strftime('%H:%M:%S')})")

df["translated_text"] = translated_texts
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"\n✅ Çeviri tamamlandı ve kaydedildi: {output_path}")
print(f"📊 Toplam çevrilen satır: {len(df)}")

✅ Veri yüklendi: 12000 satır


Translating with Google NMT:   8%|▊         | 1000/12000 [09:37<1:37:48,  1.87it/s]

💾 Ara kayıt yapıldı: 1000/12000 satır (20:00:09)


Translating with Google NMT:  17%|█▋        | 2000/12000 [19:27<1:59:42,  1.39it/s]

💾 Ara kayıt yapıldı: 2000/12000 satır (20:09:58)


...

💾 Ara kayıt yapıldı: 10000/12000 satır (21:28:31)


Translating with Google NMT:  92%|█████████▏| 11000/12000 [1:47:38<09:35,  1.74it/s]

💾 Ara kayıt yapıldı: 11000/12000 satır (21:38:09)


Translating with Google NMT: 100%|██████████| 12000/12000 [1:56:57<00:00,  1.71it/s]

💾 Ara kayıt yapıldı: 12000/12000 satır (21:47:29)

✅ Çeviri tamamlandı ve kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_google.csv
📊 Toplam çevrilen satır: 12000


# **LibreTranslate Çeviri**

In [ ]:
!pip install libretranslatepy

In [ ]:
from tqdm import tqdm
import pandas as pd
import requests
import time


input_path  = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced.csv"
output_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_libre_fixed.csv"


df = pd.read_csv(input_path)
print(f"✅ Veri yüklendi: {len(df)} satır")

url = "https://translate.fedilab.app/translate"

def libre_translate(text, source_lang, target_lang="en"):
    payload = {
        "q": text,
        "source": source_lang,
        "target": target_lang,
        "format": "text"
    }
    try:
        r = requests.post(url, data=payload, timeout=25)
        data = r.json()
        return data.get("translatedText", "")
    except Exception as e:
        print(f"Hata: {e}")
        return ""

batch_size = 500
translated_texts = []
start_time = time.time()

for i in tqdm(range(0, len(df), batch_size), desc="Translating with LibreTranslate", ncols=100):
    batch_start = time.time()
    batch = df.iloc[i:i+batch_size]
    batch_translations = []

    for idx, (text, lang) in enumerate(zip(batch["text"], batch["language"])):
        src_lang = "zh" if lang.startswith("zh") else lang
        tr = libre_translate(text, src_lang)
        batch_translations.append(tr)

        if idx % 50 == 0 and idx != 0:
            elapsed = time.time() - start_time
            done = i + idx
            speed = done / elapsed
            remaining = (len(df) - done) / speed / 60
            print(f"⏱️ {done}/{len(df)} tamamlandı | {speed:.2f} satır/sn | Tahmini kalan: {remaining:.1f} dk", end="\r")

        time.sleep(0.7)

    translated_texts.extend(batch_translations)

    temp = df.iloc[:i+batch_size].copy()
    temp["translated_text"] = translated_texts
    temp.to_csv(output_path, index=False, encoding="utf-8")
    print(f"\n💾 Ara kayıt yapıldı: {i+batch_size}/{len(df)} satır ({time.strftime('%H:%M:%S')})")

df["translated_text"] = translated_texts
df.to_csv(output_path, index=False, encoding="utf-8")

elapsed_total = (time.time() - start_time) / 60
print(f"\n✅ Çeviri tamamlandı ve kaydedildi: {output_path}")
print(f"📊 Toplam çevrilen satır: {len(df)}")
print(f"⏰ Toplam süre: {elapsed_total:.1f} dakika")

✅ Veri yüklendi: 12000 satır


Translating with LibreTranslate:   0%|                                       | 0/24 [00:00<?, ?it/s]

Translating with LibreTranslate:   4%|█▏                          | 1/24 [13:41<5:15:02, 821.86s/it]


💾 Ara kayıt yapıldı: 500/12000 satır (09:56:25)


Translating with LibreTranslate:   8%|██▎                         | 2/24 [27:20<5:00:37, 819.89s/it]

...


💾 Ara kayıt yapıldı: 11000/12000 satır (14:44:53)


Translating with LibreTranslate:  96%|█████████████████████████▉ | 23/24 [5:15:54<13:44, 824.54s/it]


💾 Ara kayıt yapıldı: 11500/12000 satır (14:58:38)


Translating with LibreTranslate: 100%|███████████████████████████| 24/24 [5:29:39<00:00, 824.13s/it]


💾 Ara kayıt yapıldı: 12000/12000 satır (15:12:23)

✅ Çeviri tamamlandı ve kaydedildi: /content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_libre_fixed.csv
📊 Toplam çevrilen satır: 12000
⏰ Toplam süre: 329.7 dakika


# **Boş Veri Temizleme**

In [ ]:
for df in [libre, google, mbart]:
    df.dropna(subset=['translated_text'], inplace=True)
    df['translated_text'] = df['translated_text'].astype(str)

In [ ]:
for name, df in [("Libre", libre), ("Google", google), ("mBART", mbart)]:
    print(f"\n{name} veri durumu:")
    print(df['translated_text'].isnull().sum(), "adet boş satır var")
    print(df['translated_text'].apply(type).value_counts().head())


Libre veri durumu:
0 adet boş satır var
translated_text
<class 'str'>    12000
Name: count, dtype: int64

Google veri durumu:
0 adet boş satır var
translated_text
<class 'str'>    8997
Name: count, dtype: int64

mBART veri durumu:
0 adet boş satır var
translated_text
<class 'str'>    12000
Name: count, dtype: int64


# **GoogleTranslate + RoBERTa**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_google_clean.csv")
print("✅ Veri seti yüklendi:", df.shape)

✅ Veri seti yüklendi: (11997, 4)


In [ ]:
missing_translations = df["translated_text"].isna().sum()
empty_translations = (df["translated_text"].astype(str).str.strip() == "").sum()

print(f"❌ NaN (eksik) çeviri sayısı: {missing_translations}")
print(f"⚠️ Boş string ('' veya ' ') çeviri sayısı: {empty_translations}")
print(f"🧮 Toplam boş çeviri sayısı: {missing_translations + empty_translations}")

❌ NaN (eksik) çeviri sayısı: 0
⚠️ Boş string ('' veya ' ') çeviri sayısı: 0
🧮 Toplam boş çeviri sayısı: 0


In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn -q

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset

df = pd.read_csv("/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_google_clean.csv")

df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
df["label"] = df["sentiment"].map(label_map)

print("✅ Veri hazır:", df.shape)

✅ Veri hazır: (11997, 5)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize(batch):
    return tokenizer(batch["translated_text"], truncation=True, padding="max_length", max_length=128)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1_score}

# K-Fold ayarları
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} başlıyor...")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = Dataset.from_pandas(train_df[["translated_text", "label"]])
    val_dataset   = Dataset.from_pandas(val_df[["translated_text", "label"]])

    train_enc = train_dataset.map(tokenize, batched=True)
    val_enc   = val_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"/content/roberta_kfold/fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"/content/logs/fold_{fold}",
        report_to="none",
        logging_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_enc,
        eval_dataset=val_enc,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()

    metrics = trainer.evaluate()
    print(f"📊 Fold {fold} sonuçları:", metrics)
    fold_results.append(metrics)

print("\n✅ Tüm fold'lar tamamlandı.\n")


📘 Fold 1 başlıyor...


Map:   0%|          | 0/9597 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

/tmp/ipython-input-2941609588.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Map:   0%|          | 0/9598 [00:00<?, ? examples/s]

Map:   0%|          | 0/2399 [00:00<?, ? examples/s]

/tmp/ipython-input-2941609588.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.747200,0.770781,0.686119,0.667370
2,0.620500,0.748807,0.691538,0.691687
3,0.452400,0.820590,0.702793,0.701826


📊 Fold 5 sonuçları: {'eval_loss': 0.8205898404121399, 'eval_accuracy': 0.7027928303459775, 'eval_f1': 0.7018264442174478, 'eval_runtime': 7.2008, 'eval_samples_per_second': 333.156, 'eval_steps_per_second': 10.415, 'epoch': 3.0}

✅ Tüm fold'lar tamamlandı.



In [ ]:
import numpy as np

avg_acc = np.mean([r["eval_accuracy"] for r in fold_results])
avg_f1 = np.mean([r["eval_f1"] for r in fold_results])

print(f"\n✅ Ortalama sonuçlar (5-Fold Cross Validation):")
print(f"Accuracy: {avg_acc:.4f}")
print(f"F1-score: {avg_f1:.4f}")


✅ Ortalama sonuçlar (5-Fold Cross Validation):
Accuracy: 0.7045
F1-score: 0.7012


# **GoogleTranslate + BERTweet**

In [ ]:
from transformers import RobertaTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
import numpy as np, evaluate, pandas as pd

model_name = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, normalization=True)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def tokenize(batch):
    return tokenizer(batch["translated_text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1_score}

df = pd.read_csv("/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_google_clean.csv")
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
df["label"] = df["sentiment"].map(label_map)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} başlıyor...")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = Dataset.from_pandas(train_df[["translated_text", "label"]])
    val_dataset = Dataset.from_pandas(val_df[["translated_text", "label"]])
    train_enc = train_dataset.map(tokenize, batched=True)
    val_enc = val_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"/content/bertweet_kfold/fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"/content/logs/bertweet_fold_{fold}",
        report_to="none",
        logging_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_enc,
        eval_dataset=val_enc,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    metrics = trainer.evaluate()
    print(f"📊 Fold {fold} sonuçları:", metrics)
    fold_results.append(metrics)

avg_acc = np.mean([r["eval_accuracy"] for r in fold_results])
avg_f1 = np.mean([r["eval_f1"] for r in fold_results])

print(f"\n✅ Ortalama sonuçlar (BERTweet, 5-Fold):\nAccuracy: {avg_acc:.4f}\nF1-score: {avg_f1:.4f}")

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


...

Map:   0%|          | 0/9598 [00:00<?, ? examples/s]

Map:   0%|          | 0/2399 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-44706307.py:61: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.869700,0.771972,0.677782,0.653571
2,0.650500,0.717792,0.702376,0.703071
3,0.494000,0.770379,0.696123,0.696356


📊 Fold 5 sonuçları: {'eval_loss': 0.7177923917770386, 'eval_accuracy': 0.7023759899958316, 'eval_f1': 0.7030705962732209, 'eval_runtime': 7.2887, 'eval_samples_per_second': 329.138, 'eval_steps_per_second': 10.29, 'epoch': 3.0}

✅ Ortalama sonuçlar (BERTweet, 5-Fold):
Accuracy: 0.6988
F1-score: 0.6971


# **GoogleTranslate + BERTweet Classification Report**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from glob import glob
import numpy as np
import torch

label_names = ["NEG", "NEUTRAL", "POS"]

print("🔎 BERTweet fine-tuning sonrası classification raporları hazırlanıyor...")

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} classification report:")

    checkpoints = sorted(glob(f"/content/bertweet_kfold/fold_{fold}/checkpoint-*"))
    if not checkpoints:
        print(f"⚠️ Fold {fold} için checkpoint bulunamadı, atlanıyor.")
        continue

    latest_checkpoint = checkpoints[-1]
    print(f"📂 Yüklenen checkpoint: {latest_checkpoint}")

    model = AutoModelForSequenceClassification.from_pretrained(latest_checkpoint)
    tokenizer = AutoTokenizer.from_pretrained(latest_checkpoint)

    model.eval()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    val_df = df.iloc[val_idx]
    val_texts = val_df["translated_text"].tolist()
    val_labels = val_df["label"].tolist()

    preds = []
    batch_size = 32
    device = model.device

    for i in range(0, len(val_texts), batch_size):
        batch = val_texts[i:i + batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        preds.extend(np.argmax(outputs.logits.detach().cpu().numpy(), axis=-1))

    print(classification_report(val_labels, preds, target_names=label_names, digits=3))

🔎 BERTweet fine-tuning sonrası classification raporları hazırlanıyor...

📘 Fold 1 classification report:
📂 Yüklenen checkpoint: /content/bertweet_kfold/fold_1/checkpoint-600


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


              precision    recall  f1-score   support

         NEG      0.758     0.699     0.727       973
     NEUTRAL      0.465     0.387     0.423       659
         POS      0.673     0.837     0.746       768

    accuracy                          0.657      2400
   macro avg      0.632     0.641     0.632      2400
weighted avg      0.651     0.657     0.650      2400


📘 Fold 2 classification report:
📂 Yüklenen checkpoint: /content/bertweet_kfold/fold_2/checkpoint-600


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


              precision    recall  f1-score   support

         NEG      0.724     0.826     0.772       973
     NEUTRAL      0.560     0.410     0.473       659
         POS      0.747     0.785     0.766       768

    accuracy                          0.699      2400
   macro avg      0.677     0.674     0.670      2400
weighted avg      0.686     0.699     0.688      2400


📘 Fold 3 classification report:
📂 Yüklenen checkpoint: /content/bertweet_kfold/fold_3/checkpoint-600


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


              precision    recall  f1-score   support

         NEG      0.747     0.781     0.763       973
     NEUTRAL      0.517     0.413     0.459       659
         POS      0.729     0.812     0.768       767

    accuracy                          0.690      2399
   macro avg      0.664     0.669     0.664      2399
weighted avg      0.678     0.690     0.681      2399


📘 Fold 4 classification report:
📂 Yüklenen checkpoint: /content/bertweet_kfold/fold_4/checkpoint-600


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


              precision    recall  f1-score   support

         NEG      0.684     0.861     0.763       973
     NEUTRAL      0.563     0.299     0.390       659
         POS      0.752     0.808     0.779       767

    accuracy                          0.690      2399
   macro avg      0.666     0.656     0.644      2399
weighted avg      0.673     0.690     0.666      2399


📘 Fold 5 classification report:
📂 Yüklenen checkpoint: /content/bertweet_kfold/fold_5/checkpoint-600
              precision    recall  f1-score   support

         NEG      0.726     0.794     0.759       972
     NEUTRAL      0.550     0.291     0.381       659
         POS      0.671     0.862     0.754       768

    accuracy                          0.678      2399
   macro avg      0.649     0.649     0.631      2399
weighted avg      0.660     0.678     0.654      2399



# **GoogleTranslate + RoBERTa Classification Report**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from glob import glob
import numpy as np
import torch

label_names = ["NEG", "NEUTRAL", "POS"]

print("🔎 Twitter-RoBERTa fine-tuning sonrası classification raporları hazırlanıyor...")

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} classification report:")

    checkpoints = sorted(glob(f"/content/roberta_kfold/fold_{fold}/checkpoint-*"))
    if not checkpoints:
        print(f"⚠️ Fold {fold} için checkpoint bulunamadı, atlanıyor.")
        continue

    latest_checkpoint = checkpoints[-1]
    print(f"📂 Yüklenen checkpoint: {latest_checkpoint}")

    model = AutoModelForSequenceClassification.from_pretrained(latest_checkpoint)
    tokenizer = AutoTokenizer.from_pretrained(latest_checkpoint)

    model.eval()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    val_df = df.iloc[val_idx]
    val_texts = val_df["translated_text"].tolist()
    val_labels = val_df["label"].tolist()

    preds = []
    batch_size = 32
    device = model.device

    for i in range(0, len(val_texts), batch_size):
        batch = val_texts[i:i + batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        preds.extend(np.argmax(outputs.logits.detach().cpu().numpy(), axis=-1))

    print(classification_report(val_labels, preds, target_names=label_names, digits=3))

🔎 Twitter-RoBERTa fine-tuning sonrası classification raporları hazırlanıyor...

📘 Fold 1 classification report:
📂 Yüklenen checkpoint: /content/roberta_kfold/fold_1/checkpoint-600
              precision    recall  f1-score   support

         NEG      0.730     0.793     0.760       973
     NEUTRAL      0.523     0.507     0.515       659
         POS      0.812     0.743     0.776       768

    accuracy                          0.699      2400
   macro avg      0.688     0.681     0.684      2400
weighted avg      0.699     0.699     0.698      2400


📘 Fold 2 classification report:
📂 Yüklenen checkpoint: /content/roberta_kfold/fold_2/checkpoint-600
              precision    recall  f1-score   support

         NEG      0.711     0.855     0.776       973
     NEUTRAL      0.563     0.407     0.472       659
         POS      0.768     0.753     0.760       768

    accuracy                          0.699      2400
   macro avg      0.680     0.671     0.669      2400
weighted avg

In [ ]:
import shutil, os

base_models = [
    "roberta_kfold",
    "bertweet_kfold"
]

save_root = "/content/drive/MyDrive/NLP/Translation/final_models"
os.makedirs(save_root, exist_ok=True)

for model_name in base_models:
    model_path = f"/content/{model_name}"
    final_path = os.path.join(save_root, model_name.replace("_kfold", "_final"))

    best_ckpt = None
    best_step = 0

    for root, dirs, files in os.walk(model_path):
        for d in dirs:
            if d.startswith("checkpoint-"):
                step = int(d.split("-")[1])
                if step > best_step:
                    best_step = step
                    best_ckpt = os.path.join(root, d)

    if best_ckpt:
        print(f"📦 {model_name} -> {best_ckpt} (step {best_step})")
        shutil.copytree(best_ckpt, final_path)
        print(f"✅ {model_name.replace('_kfold','_final')} kaydedildi.\n")
    else:
        print(f"⚠️ {model_name} için checkpoint bulunamadı.\n")

📦 roberta_kfold -> /content/roberta_kfold/fold_2/checkpoint-1800 (step 1800)
✅ roberta_final kaydedildi.

📦 bertweet_kfold -> /content/bertweet_kfold/fold_5/checkpoint-1800 (step 1800)
✅ bertweet_final kaydedildi.



# **LibreTranslate + RoBERTa**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_libre_fixed.csv")
print("✅ Veri seti yüklendi:", df.shape)

✅ Veri seti yüklendi: (12000, 4)


In [ ]:
missing_translations = df["translated_text"].isna().sum()
empty_translations = (df["translated_text"].astype(str).str.strip() == "").sum()

print(f"❌ NaN (eksik) çeviri sayısı: {missing_translations}")
print(f"⚠️ Boş string ('' veya ' ') çeviri sayısı: {empty_translations}")
print(f"🧮 Toplam boş çeviri sayısı: {missing_translations + empty_translations}")

❌ NaN (eksik) çeviri sayısı: 0
⚠️ Boş string ('' veya ' ') çeviri sayısı: 0
🧮 Toplam boş çeviri sayısı: 0


In [ ]:
!pip install -U transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 63.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
import numpy as np, evaluate, pandas as pd

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_libre_fixed.csv"

df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
df["label"] = df["sentiment"].map(label_map)

print(f"✅ Libre veri seti hazır: {df.shape}")

model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def tokenize(batch):
    return tokenizer(batch["translated_text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1_score}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} başlıyor...")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = Dataset.from_pandas(train_df[["translated_text", "label"]])
    val_dataset = Dataset.from_pandas(val_df[["translated_text", "label"]])

    train_enc = train_dataset.map(tokenize, batched=True)
    val_enc = val_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"/content/roberta_libre_kfold/fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"/content/logs/roberta_libre_fold_{fold}",
        report_to="none",
        logging_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_enc,
        eval_dataset=val_enc,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    metrics = trainer.evaluate()
    print(f"📊 Fold {fold} sonuçları:", metrics)
    fold_results.append(metrics)

avg_acc = np.mean([r["eval_accuracy"] for r in fold_results])
avg_f1 = np.mean([r["eval_f1"] for r in fold_results])

print(f"\n✅ Ortalama sonuçlar (RoBERTa + Libre, 5-Fold):\nAccuracy: {avg_acc:.4f}\nF1-score: {avg_f1:.4f}")

✅ Libre veri seti hazır: (12000, 5)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

/tmp/ipython-input-4030974419.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.837500,0.784828,0.644583,0.642091
2,0.700500,0.805104,0.652083,0.647273
3,0.522100,0.900191,0.656250,0.654378


📊 Fold 5 sonuçları: {'eval_loss': 0.9001909494400024, 'eval_accuracy': 0.65625, 'eval_f1': 0.6543780144182402, 'eval_runtime': 7.1567, 'eval_samples_per_second': 335.352, 'eval_steps_per_second': 10.48, 'epoch': 3.0}

✅ Ortalama sonuçlar (RoBERTa + Libre, 5-Fold):
Accuracy: 0.6563
F1-score: 0.6540


# **LibreTranslate + BERTweet**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
import numpy as np, evaluate, pandas as pd

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_libre_fixed.csv"

df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
df["label"] = df["sentiment"].map(label_map)

print(f"✅ Libre veri seti hazır: {df.shape}")

model_name = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, normalization=True)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def tokenize(batch):
    return tokenizer(batch["translated_text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1_score}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} başlıyor...")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = Dataset.from_pandas(train_df[["translated_text", "label"]])
    val_dataset = Dataset.from_pandas(val_df[["translated_text", "label"]])

    train_enc = train_dataset.map(tokenize, batched=True)
    val_enc = val_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"/content/bertweet_libre_kfold/fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"/content/logs/bertweet_libre_fold_{fold}",
        report_to="none",
        logging_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_enc,
        eval_dataset=val_enc,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    metrics = trainer.evaluate()
    print(f"📊 Fold {fold} sonuçları:", metrics)
    fold_results.append(metrics)

avg_acc = np.mean([r["eval_accuracy"] for r in fold_results])
avg_f1 = np.mean([r["eval_f1"] for r in fold_results])

print(f"\n✅ Ortalama sonuçlar (BERTweet + Libre, 5-Fold):\nAccuracy: {avg_acc:.4f}\nF1-score: {avg_f1:.4f}")

✅ Libre veri seti hazır: (12000, 5)


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2436148554.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.927900,0.796018,0.635833,0.641765
2,0.734800,0.782777,0.648750,0.650339
3,0.579800,0.846856,0.647917,0.646848


📊 Fold 5 sonuçları: {'eval_loss': 0.7827774286270142, 'eval_accuracy': 0.64875, 'eval_f1': 0.6503386068904101, 'eval_runtime': 7.2951, 'eval_samples_per_second': 328.99, 'eval_steps_per_second': 10.281, 'epoch': 3.0}

✅ Ortalama sonuçlar (BERTweet + Libre, 5-Fold):
Accuracy: 0.6557
F1-score: 0.6548


# **LibreTranslate + BERTweet Classification Report**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from datasets import Dataset
import pandas as pd, numpy as np, glob, torch

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_libre_fixed.csv"
df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
inv_label_map = {v: k for k, v in label_map.items()}
df["label"] = df["sentiment"].map(label_map)

model_name = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, normalization=True)

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_reports = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} raporu hazırlanıyor...")

    val_df = df.iloc[val_idx]
    val_texts = val_df["translated_text"].tolist()
    val_labels = val_df["label"].tolist()

    checkpoints = sorted(glob.glob(f"/content/bertweet_libre_kfold/fold_{fold}/checkpoint-*"))
    if not checkpoints:
        print(f"❌ Fold {fold} için checkpoint bulunamadı.")
        continue
    best_ckpt = checkpoints[-1]

    model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)
    model.to(device)
    model.eval()

    inputs = tokenizer(val_texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()

    report = classification_report(val_labels, preds, target_names=["NEG", "NEUTRAL", "POS"], output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    print(df_report)
    fold_reports.append(df_report)

print("\n\n✅ Ortalama Rapor (Tüm Fold'ların Ortalaması):")
avg_report = sum(fold_reports) / len(fold_reports)
print(avg_report)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0



📘 Fold 1 raporu hazırlanıyor...
              precision    recall  f1-score    support
NEG            0.693857  0.708119  0.700916   973.0000
NEUTRAL        0.482072  0.367223  0.416882   659.0000
POS            0.648619  0.764323  0.701733   768.0000
accuracy       0.632500  0.632500  0.632500     0.6325
macro avg      0.608182  0.613222  0.606510  2400.0000
weighted avg   0.621228  0.632500  0.623186  2400.0000

📘 Fold 2 raporu hazırlanıyor...
              precision    recall  f1-score    support
NEG            0.648316  0.810894  0.720548   973.0000
NEUTRAL        0.485944  0.367223  0.418323   659.0000
POS            0.763504  0.680990  0.719890   768.0000
accuracy       0.647500  0.647500  0.647500     0.6475
macro avg      0.632588  0.619702  0.619587  2400.0000
weighted avg   0.640591  0.647500  0.637352  2400.0000

📘 Fold 3 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.642114  0.824255  0.721872   973.000000
NEUTRAL        0.

# **LibreTranslate + RoBERTa Classification Report**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from datasets import Dataset
import pandas as pd, numpy as np, glob, torch
from sklearn.model_selection import StratifiedKFold

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_libre_fixed.csv"
df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
inv_label_map = {v: k for k, v in label_map.items()}
df["label"] = df["sentiment"].map(label_map)

model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} raporu hazırlanıyor...")

    val_df = df.iloc[val_idx]
    val_texts = val_df["translated_text"].tolist()
    val_labels = val_df["label"].tolist()

    checkpoints = sorted(glob.glob(f"/content/roberta_libre_kfold/fold_{fold}/checkpoint-*"))
    if not checkpoints:
        print(f"❌ Fold {fold} için checkpoint bulunamadı.")
        continue
    best_ckpt = checkpoints[-1]

    model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)
    model.to(device)
    model.eval()

    inputs = tokenizer(val_texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()

    report = classification_report(val_labels, preds, target_names=["NEG", "NEUTRAL", "POS"], output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    print(df_report)
    fold_reports.append(df_report)

print("\n\n✅ Ortalama Rapor (RoBERTa + Libre, 5-Fold):")
avg_report = sum(fold_reports) / len(fold_reports)
print(avg_report)


📘 Fold 1 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.703307  0.743063  0.722639   973.000000
NEUTRAL        0.493506  0.403642  0.444073   659.000000
POS            0.693878  0.752604  0.722049   768.000000
accuracy       0.652917  0.652917  0.652917     0.652917
macro avg      0.630230  0.633103  0.629587  2400.000000
weighted avg   0.642682  0.652917  0.645961  2400.000000

📘 Fold 2 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.703048  0.734841  0.718593   973.000000
NEUTRAL        0.473541  0.529590  0.500000   659.000000
POS            0.792570  0.666667  0.724187   768.000000
accuracy       0.656667  0.656667  0.656667     0.656667
macro avg      0.656386  0.643699  0.647593  2400.000000
weighted avg   0.668676  0.656667  0.660361  2400.000000

📘 Fold 3 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.695985  0.748201  0.721149   

# **mBART + RoBERTa**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_mbart_clean.csv")
print("✅ Veri seti yüklendi:", df.shape)

✅ Veri seti yüklendi: (11998, 4)


In [ ]:
missing_translations = df["translated_text"].isna().sum()
empty_translations = (df["translated_text"].astype(str).str.strip() == "").sum()

print(f"❌ NaN (eksik) çeviri sayısı: {missing_translations}")
print(f"⚠️ Boş string ('' veya ' ') çeviri sayısı: {empty_translations}")
print(f"🧮 Toplam boş çeviri sayısı: {missing_translations + empty_translations}")

❌ NaN (eksik) çeviri sayısı: 0
⚠️ Boş string ('' veya ' ') çeviri sayısı: 0
🧮 Toplam boş çeviri sayısı: 0


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
import numpy as np, evaluate, pandas as pd

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_mbart_clean.csv"

df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
df["label"] = df["sentiment"].map(label_map)
print(f"✅ mBART veri seti hazır: {df.shape}")

model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def tokenize(batch):
    return tokenizer(batch["translated_text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1_score}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} başlıyor...")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = Dataset.from_pandas(train_df[["translated_text", "label"]])
    val_dataset = Dataset.from_pandas(val_df[["translated_text", "label"]])

    train_enc = train_dataset.map(tokenize, batched=True)
    val_enc = val_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"/content/roberta_mbart_kfold/fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"/content/logs/roberta_mbart_fold_{fold}",
        report_to="none",
        logging_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_enc,
        eval_dataset=val_enc,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    metrics = trainer.evaluate()
    print(f"📊 Fold {fold} sonuçları:", metrics)
    fold_results.append(metrics)

avg_acc = np.mean([r["eval_accuracy"] for r in fold_results])
avg_f1 = np.mean([r["eval_f1"] for r in fold_results])

print(f"\n✅ Ortalama sonuçlar (RoBERTa + mBART, 5-Fold):\nAccuracy: {avg_acc:.4f}\nF1-score: {avg_f1:.4f}")

✅ mBART veri seti hazır: (11998, 5)

📘 Fold 1 başlıyor...


Map:   0%|          | 0/9598 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

/tmp/ipython-input-2426449462.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.923400,0.838733,0.599583,0.567563
2,0.782100,0.837521,0.618750,0.606355
3,0.604300,0.917531,0.619167,0.617894


...

Map:   0%|          | 0/9599 [00:00<?, ? examples/s]

Map:   0%|          | 0/2399 [00:00<?, ? examples/s]

/tmp/ipython-input-2426449462.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.905000,0.857166,0.601917,0.601102
2,0.778600,0.861759,0.616507,0.610761
3,0.605000,0.963749,0.614840,0.608676


📊 Fold 5 sonuçları: {'eval_loss': 0.8617594242095947, 'eval_accuracy': 0.6165068778657774, 'eval_f1': 0.6107609174952219, 'eval_runtime': 7.2087, 'eval_samples_per_second': 332.792, 'eval_steps_per_second': 10.404, 'epoch': 3.0}

✅ Ortalama sonuçlar (RoBERTa + mBART, 5-Fold):
Accuracy: 0.6167
F1-score: 0.6129


# **mBART + BERTweet**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
import numpy as np, evaluate, pandas as pd


data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_mbart_clean.csv"

df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
df["label"] = df["sentiment"].map(label_map)
print(f"✅ mBART veri seti hazır: {df.shape}")

model_name = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def tokenize(batch):
    return tokenizer(batch["translated_text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1_score}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} başlıyor...")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = Dataset.from_pandas(train_df[["translated_text", "label"]])
    val_dataset = Dataset.from_pandas(val_df[["translated_text", "label"]])

    train_enc = train_dataset.map(tokenize, batched=True)
    val_enc = val_dataset.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=f"/content/bertweet_mbart_kfold/fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"/content/logs/bertweet_mbart_fold_{fold}",
        report_to="none",
        logging_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_enc,
        eval_dataset=val_enc,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    metrics = trainer.evaluate()
    print(f"📊 Fold {fold} sonuçları:", metrics)
    fold_results.append(metrics)

avg_acc = np.mean([r["eval_accuracy"] for r in fold_results])
avg_f1 = np.mean([r["eval_f1"] for r in fold_results])

print(f"\n✅ Ortalama sonuçlar (BERTweet + mBART, 5-Fold):\nAccuracy: {avg_acc:.4f}\nF1-score: {avg_f1:.4f}")

✅ mBART veri seti hazır: (11998, 5)


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0



📘 Fold 1 başlıyor...


Map:   0%|          | 0/9598 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/9599 [00:00<?, ? examples/s]

Map:   0%|          | 0/2399 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3340743899.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.984600,0.882777,0.583160,0.583244
2,0.814400,0.872442,0.608170,0.600859
3,0.675800,0.891710,0.608587,0.608061


📊 Fold 5 sonuçları: {'eval_loss': 0.8917098641395569, 'eval_accuracy': 0.6085869112130055, 'eval_f1': 0.6080610948114704, 'eval_runtime': 7.2914, 'eval_samples_per_second': 329.018, 'eval_steps_per_second': 10.286, 'epoch': 3.0}

✅ Ortalama sonuçlar (BERTweet + mBART, 5-Fold):
Accuracy: 0.6129
F1-score: 0.6129


# **mBART + RoBERTa Classification Report**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from datasets import Dataset
import pandas as pd, numpy as np, glob, torch
from sklearn.model_selection import StratifiedKFold

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_mbart_clean.csv"
df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
inv_label_map = {v: k for k, v in label_map.items()}
df["label"] = df["sentiment"].map(label_map)

model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} raporu hazırlanıyor...")

    val_df = df.iloc[val_idx]
    val_texts = val_df["translated_text"].tolist()
    val_labels = val_df["label"].tolist()

    checkpoints = sorted(glob.glob(f"/content/roberta_mbart_kfold/fold_{fold}/checkpoint-*"))
    if not checkpoints:
        print(f"❌ Fold {fold} için checkpoint bulunamadı.")
        continue
    best_ckpt = checkpoints[-1]

    model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)
    model.to(device)
    model.eval()

    inputs = tokenizer(
        val_texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()

    report = classification_report(
        val_labels,
        preds,
        target_names=["NEG", "NEUTRAL", "POS"],
        output_dict=True
    )
    df_report = pd.DataFrame(report).transpose()
    print(df_report)
    fold_reports.append(df_report)

print("\n\n✅ Ortalama Rapor (RoBERTa + mBART, 5-Fold):")
avg_report = sum(fold_reports) / len(fold_reports)
print(avg_report)


📘 Fold 1 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.620285  0.760534  0.683287   973.000000
NEUTRAL        0.444056  0.192716  0.268783   659.000000
POS            0.621064  0.744792  0.677324   768.000000
accuracy       0.599583  0.599583  0.599583     0.599583
macro avg      0.561802  0.566014  0.543131  2400.000000
weighted avg   0.572145  0.599583  0.567563  2400.000000

📘 Fold 2 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.552856  0.865365  0.674679   973.000000
NEUTRAL        0.435897  0.180577  0.255365   659.000000
POS            0.711921  0.559896  0.626822   768.000000
accuracy       0.579583  0.579583  0.579583     0.579583
macro avg      0.566891  0.535279  0.518955  2400.000000
weighted avg   0.571642  0.579583  0.544228  2400.000000

📘 Fold 3 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.580761  0.832305  0.684144   

# **mBART + BERTweet Classification Report**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
import pandas as pd, numpy as np, glob, torch

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_mbart_clean.csv"
df = pd.read_csv(data_path)
df.dropna(subset=["translated_text", "sentiment"], inplace=True)
df["translated_text"] = df["translated_text"].astype(str)

label_map = {"NEG": 0, "NEUTRAL": 1, "POS": 2}
df["label"] = df["sentiment"].map(label_map)

model_name = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_reports = []
base_path = "/content/bertweet_mbart_kfold"

for fold, (train_idx, val_idx) in enumerate(skf.split(df["translated_text"], df["label"]), 1):
    print(f"\n📘 Fold {fold} raporu hazırlanıyor...")

    val_df = df.iloc[val_idx]
    val_texts = val_df["translated_text"].tolist()
    val_labels = val_df["label"].tolist()

    checkpoints = sorted(glob.glob(f"{base_path}/fold_{fold}/checkpoint-*"))
    if not checkpoints:
        print(f"❌ Fold {fold} için checkpoint bulunamadı.")
        continue
    best_ckpt = checkpoints[-1]

    model = AutoModelForSequenceClassification.from_pretrained(best_ckpt).to(device)
    model.eval()

    inputs = tokenizer(val_texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()

    report = classification_report(val_labels, preds, target_names=["NEG", "NEUTRAL", "POS"], output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    print(df_report)
    fold_reports.append(df_report)

print("\n\n✅ Ortalama Rapor (BERTweet + mBART, 5-Fold):")
avg_report = sum(fold_reports) / len(fold_reports)
print(avg_report)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0



📘 Fold 1 raporu hazırlanıyor...
              precision    recall  f1-score    support
NEG            0.554702  0.891059  0.683754   973.0000
NEUTRAL        0.472868  0.092564  0.154822   659.0000
POS            0.697740  0.643229  0.669377   768.0000
accuracy       0.592500  0.592500  0.592500     0.5925
macro avg      0.575104  0.542284  0.502651  2400.0000
weighted avg   0.578004  0.592500  0.533917  2400.0000

📘 Fold 2 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.532931  0.906475  0.671233   973.000000
NEUTRAL        0.459227  0.162367  0.239910   659.000000
POS            0.755859  0.503906  0.604688   768.000000
accuracy       0.573333  0.573333  0.573333     0.573333
macro avg      0.582672  0.524249  0.505277  2400.000000
weighted avg   0.584030  0.573333  0.531504  2400.000000

📘 Fold 3 raporu hazırlanıyor...
              precision    recall  f1-score      support
NEG            0.561333  0.866255  0.681230   972.000000
NEU

In [ ]:
import shutil, os

base_models = [
    "roberta_mbart_kfold",
    "bertweet_mbart_kfold"
]

save_root = "/content/drive/MyDrive/NLP/Translation/final_models"
os.makedirs(save_root, exist_ok=True)

for model_name in base_models:
    model_path = f"/content/{model_name}"
    final_path = os.path.join(save_root, model_name.replace("_kfold", "_final"))

    best_ckpt = None
    best_step = 0

    for root, dirs, files in os.walk(model_path):
        for d in dirs:
            if d.startswith("checkpoint-"):
                step = int(d.split("-")[1])
                if step > best_step:
                    best_step = step
                    best_ckpt = os.path.join(root, d)

    if best_ckpt:
        print(f"📦 {model_name} -> {best_ckpt} (step {best_step})")
        shutil.copytree(best_ckpt, final_path)
        print(f"✅ {model_name.replace('_kfold','_final')} kaydedildi.\n")
    else:
        print(f"⚠️ {model_name} için checkpoint bulunamadı.\n")

📦 roberta_mbart_kfold -> /content/roberta_mbart_kfold/fold_2/checkpoint-1800 (step 1800)
✅ roberta_mbart_final kaydedildi.

📦 bertweet_mbart_kfold -> /content/bertweet_mbart_kfold/fold_2/checkpoint-1800 (step 1800)
✅ bertweet_mbart_final kaydedildi.

